In [ ]:
import websocket
import json
import os
import threading
import time
from collections import namedtuple
from datetime import datetime
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

SYMBOL = 'ETH'
WEBSOCKET_URL = 'wss://api.hyperliquid.xyz/ws'
PARQUET_FILE = 'hyperliquid_orderbook.parquet'

OrderBook = namedtuple("OrderBook", ['ts', 'best_bid_px', 'best_bid_sz', 'best_ask_px', 'best_ask_sz'])

data_buffer = []
buffer_size = 100  
latest_prices = None

def write_to_parquet():
    if not data_buffer:
        return
    
    try:
        df = pd.DataFrame(data_buffer)
        
        data_buffer.clear()
        
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        
        schema = pa.schema([
            pa.field('timestamp', pa.timestamp('ns')),
            pa.field('hl_best_bid_px', pa.float64()),
            pa.field('hl_best_bid_sz', pa.float64()),
            pa.field('hl_best_ask_px', pa.float64()),
            pa.field('hl_best_ask_sz', pa.float64())
        ])
        
        table = pa.Table.from_pandas(df, schema=schema)
        
        if os.path.exists(PARQUET_FILE):
            existing = pq.read_table(PARQUET_FILE)
            combined = pa.concat_tables([existing, table])
            pq.write_table(combined, PARQUET_FILE)
        else:
            pq.write_table(table, PARQUET_FILE)
            
        print(f"Written {len(df)} records to {PARQUET_FILE}")
        
    except Exception as e:
        print(f"Error writing to parquet: {e}")
        data_buffer.extend(df.to_dict('records'))  # Recover data

def get_bba(inner_data):
    if 'levels' not in inner_data:
        return None  # Skip if not a book update

    bids = inner_data['levels'][0]
    asks = inner_data['levels'][1]
    
    if not bids or not asks:
        return None

    best_bid_px = float(bids[0]['px'])
    best_bid_sz = float(bids[0]['sz'])

    best_ask_px = float(asks[0]['px'])
    best_ask_sz = float(asks[0]['sz'])

    ts = datetime.fromtimestamp(inner_data['time'] / 1000)
    
    order_book = OrderBook(ts, best_bid_px, best_bid_sz, best_ask_px, best_ask_sz)
    return order_book

def on_message(ws, message):
    global latest_prices
    
    data = json.loads(message)
    if data.get('channel') != 'l2Book':
        return  # Skip if not the right channel
    
    inner_data = data.get('data', {})
    order_book = get_bba(inner_data)
    if order_book is None:
        return

    latest_prices = {
        'best_bid_px': order_book.best_bid_px,
        'best_bid_sz': order_book.best_bid_sz,
        'best_ask_px': order_book.best_ask_px,
        'best_ask_sz': order_book.best_ask_sz,
        'server_ts': order_book.ts  # Keep server timestamp if needed
    }
    
    # Print only when actual update received
    print(f"Update received: {order_book}")

def on_open(ws):
    print(f"WebSocket connected to {WEBSOCKET_URL}")
    subscription = {
        "method": "subscribe",
        "subscription": {
            "type": "l2Book",
            "coin": SYMBOL
        }
    }
    ws.send(json.dumps(subscription))
    print(f"Subscribed to L2 order book for {SYMBOL}")
    print("Listening for order book updates...")

def on_close(ws, close_status_code, close_msg):
    print("Closing WebSocket connection...")
    if data_buffer:
        write_to_parquet()

def on_error(ws, error):
    print(f"WebSocket Error: {error}")

def snapshot_thread():
    global latest_prices
    while True:
        if latest_prices:
            ts = datetime.now()
            order_book = OrderBook(ts, latest_prices['best_bid_px'], latest_prices['best_bid_sz'], latest_prices['best_ask_px'], latest_prices['best_ask_sz'])
            order_book_dict = {
                "timestamp": ts,
                "hl_best_bid_px": order_book.best_bid_px,
                "hl_best_bid_sz": order_book.best_bid_sz,
                "hl_best_ask_px": order_book.best_ask_px,
                "hl_best_ask_sz": order_book.best_ask_sz
            }
            data_buffer.append(order_book_dict)
            
            print(order_book)  # Print every snapshot to show 100ms intervals
            
            if len(data_buffer) >= buffer_size:
                write_to_parquet()
        
        time.sleep(0.1)

ws = websocket.WebSocketApp(
    WEBSOCKET_URL,
    on_message=on_message,
    on_open=on_open,
    on_close=on_close,
    on_error=on_error,
)

wst = threading.Thread(target=ws.run_forever)
wst.daemon = True
wst.start()

snapshot_t = threading.Thread(target=snapshot_thread)
snapshot_t.daemon = True
snapshot_t.start()

try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\nInterrupted by user")
    if data_buffer:
        write_to_parquet()
    ws.close()